# Colab — 100k monthly **worker** (download / enrich, без discover)

Runbook: `Fetcher/docs/COLAB_20K_RUN_RU.md`

fetcher-main (`discover` + shard 0) остаётся на RunPod. Этот ноутбук — **один и тот же файл**,
копируется на 4 отдельных Colab-аккаунта; при копировании меняются только 4 поля в CONFIG
(отмечены `# <<< МЕНЯТЬ`).

**Схема на 5 шардов (main на RunPod + 4 Colab):**
| Где | `worker_shard_index` | `worker_id` | `metrics_workers` | Роль |
|-----|----------------------|-------------|--------------------|------|
| RunPod (fetcher-main) | 0 | `fetcher-main` | — | discover + `workers` |
| Colab #1 | 1 | `colab-100k-worker-1` | 9091 | `workers` |
| Colab #2 | 2 | `colab-100k-worker-2` | 9092 | `workers` |
| Colab #3 | 3 | `colab-100k-worker-3` | 9093 | `workers` |
| Colab #4 | 4 | `colab-100k-worker-4` | 9094 | `workers` |

`worker_shard_count` = **5** везде (main и все 4 Colab), иначе шардирование разъедется и будут дубли/пропуски.

**Роли (`worker_role`):**
- `workers` — download + enrich + HF upload (как main, но без discover)
- `workers-download` — только download + upload-hf-videos
- `workers-enrich` — только enrich-metadata + upload-hf-enrich

Прогресс и coord: тот же `output_dir` и HF repo `dataset_100k_monthly_shards`.

## 0. CONFIG — **измените под этот Colab**

In [ ]:
CONFIG = {
    # --- paths / identity (общие со всеми Colab) ---
    "repo_url": "https://github.com/lebedev-ilia/TrendFlowML.git",
    "fetcher": "/content/TrendFlowML/Fetcher",
    "drive_secrets": "/content/drive/MyDrive/trendflow_secrets",
    "output_dir": "/content/dataset_runs/100k-monthly",
    "drive_backup_dir": "/content/drive/MyDrive/dataset_runs/100k-monthly-worker-1",  # <<< МЕНЯТЬ: -worker-1..-worker-4
    "hf_repo_prefix": "Ilialebedev",
    "campaign_profile": "100k-monthly",
    # --- этот Colab: 1, 2, 3 или 4 (main на RunPod = 0) ---
    "worker_shard_index": 1,  # <<< МЕНЯТЬ: 1 / 2 / 3 / 4
    "worker_shard_count": 5,
    "worker_id": "colab-100k-worker-1",  # <<< МЕНЯТЬ: colab-100k-worker-1..4
    # workers | workers-download | workers-enrich
    "worker_role": "workers",
    # --- workers / HF (должны совпадать на всех 5 шардах) ---
    "interval_sec": 120,
    "parallel_colab_count": 5,
    "hf_parallel_colab_count": 5,
    "hf_commit_min_interval_seconds": 95,
    "hf_shard_upload_batch_files": 50,
    "hf_video_upload_batch_files": 100,
    "hf_enrich_upload_batch_files": 100,
    # --- download backend: yt-dlp (+ Deno) first, pytubefix fallback ---
    "download_backend": "yt_dlp_first",
    # Подтверждено живым тестом 2026-07-20/21 (после подъёма bgutil PO Token провайдера, см.
    # секцию 1b): связка web,tv — единственная, что реально скачивает видео (android/tv по
    # одному ловят LOGIN_REQUIRED/bot, web один часто без прямых URL форматов).
    "download_ytdlp_player_clients": ["web", "tv"],
    # --- download pacing ---
    "download_pause_after_success_seconds": 10,
    "download_pause_after_fail_seconds": 15,
    "download_pause_after_bot_seconds": 120,
    "download_pause_after_fast_seconds": 15,
    # --- metrics (уникальный порт на каждый Colab) ---
    "metrics_workers": 9091,  # <<< МЕНЯТЬ: 9091 / 9092 / 9093 / 9094
    "sync_interval_sec": 120,
}

import json
from pathlib import Path

OUTPUT = Path(CONFIG["output_dir"])
OUTPUT.mkdir(parents=True, exist_ok=True)

CAMPAIGN_OVERRIDE_KEYS = {
    "download_backend",
    "download_ytdlp_player_clients",
    "hf_parallel_colab_count",
    "hf_commit_min_interval_seconds",
    "hf_shard_upload_batch_files",
    "hf_video_upload_batch_files",
    "hf_enrich_upload_batch_files",
    "download_pause_after_success_seconds",
    "download_pause_after_fail_seconds",
    "download_pause_after_unavailable_seconds",
    "download_pause_after_bot_seconds",
    "download_pause_after_fast_seconds",
    "download_fast_threshold_seconds",
    "worker_id",
    "worker_shard_index",
    "worker_shard_count",
}
CONFIG_OVERRIDES_PATH = OUTPUT / f"config_overrides_{CONFIG['worker_id']}.json"
CONFIG_OVERRIDES_PATH.write_text(
    json.dumps({k: CONFIG[k] for k in CAMPAIGN_OVERRIDE_KEYS if k in CONFIG}, indent=2),
    encoding="utf-8",
)
print("worker:", CONFIG["worker_id"], "shard", CONFIG["worker_shard_index"], "/", CONFIG["worker_shard_count"])
print("role:", CONFIG["worker_role"])
print("output_dir:", OUTPUT)
print("overrides:", CONFIG_OVERRIDES_PATH)
print("backup_dir:", CONFIG["drive_backup_dir"])

## 1. Drive + git + pip

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

dest = Path("/content/TrendFlowML")
if (dest / ".git").exists():
    subprocess.run(["git", "-C", str(dest), "pull"], check=False)
else:
    subprocess.run(["git", "clone", CONFIG["repo_url"], str(dest)], check=True)

subprocess.run(["pip", "install", "-q", "-r", str(Path(CONFIG["fetcher"]) / "requirements.txt")], check=True)
subprocess.run(["pip", "install", "-q", "-U", "huggingface_hub", "yt-dlp", "pytubefix"], check=True)
print("ready")

## 1b. PO Token provider (bgutil)

Локальный сервер на 127.0.0.1:4416 — yt-dlp находит его сам (без флагов). Без него с 2026-07-19 YouTube требует Proof-of-Origin token почти везде, yt-dlp массово падает даже со свежим IP/куки. Первый запуск на новом Colab ставит Deno + собирает сервер (~30-60с).

In [ ]:
import subprocess
r = subprocess.run(
    ["bash", str(Path(CONFIG["fetcher"]) / "scripts" / "setup_pot_provider.sh"), "/content"],
    capture_output=True, text=True,
)
print(r.stdout[-3000:])
print(r.stderr[-1500:])


## 2. HF_TOKEN + cookies

Cookies для download: скопируйте `.txt` из `drive_secrets/cookies/` в `Fetcher/fetcher/dataset_collector/cookies/` или укажите путь в campaign JSON.

In [ ]:
import os
import shutil
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
except Exception:
    pass

if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    raise RuntimeError("Set Colab Secret HF_TOKEN or os.environ[\"HF_TOKEN\"]")

hf_path = Path(CONFIG["output_dir"]) / ".hf_token"
hf_path.write_text(os.environ["HF_TOKEN"].strip() + "\n", encoding="utf-8")
print("HF_TOKEN OK:", hf_path)

secrets = Path(CONFIG["drive_secrets"])
cookie_src = secrets / "cookies"
cookie_dst = Path(CONFIG["fetcher"]) / "fetcher/dataset_collector/cookies"
if cookie_src.is_dir():
    cookie_dst.mkdir(parents=True, exist_ok=True)
    for f in cookie_src.glob("*.txt"):
        shutil.copy2(f, cookie_dst / f.name)
    print("cookies copied:", len(list(cookie_dst.glob("*.txt"))), "files")
else:
    print("WARN: no cookies at", cookie_src)

## 3. Restore state с Drive (resume после обрыва)

При первом запуске — создаёт папки. При resume — подхватывает state с Drive.

In [ ]:
import subprocess
from pathlib import Path

backup = Path(CONFIG["drive_backup_dir"])
local  = Path(CONFIG["output_dir"])
local.mkdir(parents=True, exist_ok=True)

restored = []
for d in ["state", "shards", "downloads"]:
    src = backup / d
    if src.is_dir():
        dst = local / d
        dst.mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", "--ignore-existing",
                        str(src) + "/", str(dst) + "/"], check=False)
        restored.append(d)

print("Restored from Drive:", restored if restored else "ничего (первый запуск)")

## 4. Auto-sync state → Drive (каждые 2 мин)

⚠️ Запустите **один раз** — работает в фоне весь сеанс.

In [ ]:
import threading, subprocess, time
from pathlib import Path

_sync_stop = False

def _do_sync():
    local  = Path(CONFIG["output_dir"])
    backup = Path(CONFIG["drive_backup_dir"])
    backup.mkdir(parents=True, exist_ok=True)
    synced = []
    for d in ["state", "shards", "downloads"]:
        src = local / d
        if not src.is_dir():
            continue
        dst = backup / d
        dst.mkdir(parents=True, exist_ok=True)
        r = subprocess.run(
            ["rsync", "-a",
             "--exclude=*.mp4", "--exclude=*.video.tmp", "--exclude=*.audio.tmp",
             str(src) + "/", str(dst) + "/"],
            capture_output=True
        )
        if r.returncode == 0:
            synced.append(d)
    return synced

def _sync_loop():
    interval = CONFIG["sync_interval_sec"]
    while not _sync_stop:
        time.sleep(interval)
        try:
            synced = _do_sync()
            print(f"[sync {time.strftime('%H:%M:%S')}] → Drive: {synced}")
        except Exception as e:
            print(f"[sync ERROR] {e}")

_t = threading.Thread(target=_sync_loop, daemon=True, name="drive-sync")
_t.start()
print(f"Auto-sync запущен каждые {CONFIG['sync_interval_sec']}s → {CONFIG['drive_backup_dir']}")

## 5. Workers (download / enrich)

HF progress **pull** at start / **push** after each pass — resume после обрыва Colab.
Coord (`--hf-coord`) — claims/done на HF, без дублей между Colab A/B/C.

In [ ]:
import subprocess, os
from pathlib import Path

role = CONFIG["worker_role"]
if role not in ("workers", "workers-download", "workers-enrich"):
    raise ValueError(f"worker_role must be workers|workers-download|workers-enrich, got {role!r}")

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role", role,
    "--output-dir", CONFIG["output_dir"],
    "--hf-repo-prefix", CONFIG["hf_repo_prefix"],
    "--interval", str(CONFIG["interval_sec"]),
    "--metrics-port", str(CONFIG["metrics_workers"]),
    "--worker-id", CONFIG["worker_id"],
    "--worker-shard-index", str(CONFIG["worker_shard_index"]),
    "--worker-shard-count", str(CONFIG["worker_shard_count"]),
    "--parallel-colab-count", str(CONFIG["parallel_colab_count"]),
    "--config-overrides-json", str(CONFIG_OVERRIDES_PATH),
    "--hf-coord",
]
log = Path(f"/content/workers_100k_{CONFIG['worker_id']}.log")
with open(log, "w") as fh:
    p = subprocess.Popen(
        cmd,
        cwd=str(Path(CONFIG["fetcher"])),
        stdout=fh,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
print("role:", role)
print("workers pid", p.pid)
print("log:", log)
print("tail: !tail -f", log)

## 6. Status (опционально)

In [ ]:
import subprocess, os
from pathlib import Path

subprocess.run([
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", CONFIG["campaign_profile"],
    "--role", "status",
    "--output-dir", CONFIG["output_dir"],
], cwd=str(Path(CONFIG["fetcher"])), env=os.environ.copy(), check=False)

## 7. Принудительный sync

In [ ]:
import time
synced = _do_sync()
print(f"[{time.strftime('%H:%M:%S')}] Sync → Drive: {synced}")